In [ ]:
# Data Inspection
import pandas as pd
from pathlib import Path

# Define root directory and subdirectories relative to the current working directory
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"

# Use DATA_DIR for dataset path
data_path = DATA_DIR / "nsp_electricity_dataset.csv"

# Refined read_csv call with additional parameters for handling dataset structure

In [ ]:
df = pd.read_csv(
    data_path,
    parse_dates=['timestamp'],  # Parse 'timestamp' column as datetime
    index_col='timestamp',     # Set 'timestamp' as the index
    dayfirst=False,            # Ensure dates are in MM/DD/YYYY format
    low_memory=False,          # Optimize memory usage for large datasets
    na_values=['?', 'NA', ''], # Handle missing values
    dtype={                    # Specify data types for columns
        'region': 'category',
        'consumption_kwh': 'float32'
    }
)

# Display dataset information
print(f"Dataset Shape: {df.shape}")
print(f"Dataset Columns: {df.columns.tolist()}")
print(f"First 5 Rows:\n{df.head()}")
print(f"Data Types:\n{df.dtypes}")

# Check date range of timestamp column
print(f"Date Range: {df.index.min()} to {df.index.max()}")

# Check for missing values
print(f"Missing Values:\n{df.isnull().sum()}")


In [ ]:
## Loading and Inspecting the Dataset
This section loads the dataset `nsp_electricity_dataset.csv` and inspects its structure. It includes:
- Parsing the `timestamp` column as datetime.
- Setting `timestamp` as the index.
- Handling missing values and optimizing memory usage.


In [29]:
# Filter data for Halifax and prepare for Prophet
halifax_data = df[df['region'] == 'Halifax']

# Prepare data for Prophet
prophet_data = halifax_data[['consumption_kwh']].reset_index()
prophet_data.rename(columns={'timestamp': 'ds', 'consumption_kwh': 'y'}, inplace=True)

# Display the first few rows of prepared data
print(prophet_data.head())


                   ds       y
0 2015-01-01 00:00:00  455.58
1 2015-01-01 01:00:00  579.16
2 2015-01-01 02:00:00  402.78
3 2015-01-01 03:00:00  391.23
4 2015-01-01 04:00:00  430.42


In [ ]:
### Dataset Inspection Results
- **Shape**: 438,360 rows and 26 columns.
- **Columns**: Includes `region`, `hour`, `day_of_week`, `month`, `year`, `week`, `is_weekend`, `season`, `is_holiday`, `temperature_c`, `consumption_kwh`, etc.
- **Date Range**: From `2015-01-01 00:00:00` to `2024-12-31 23:00:00`.
- **Missing Values**: None across all columns.


In [ ]:
## Understanding Data Frequency - Halifax Region Analysis
This section focuses on analyzing the data frequency for the Halifax region. It includes:
- Filtering data for Halifax.
- Calculating time differences between consecutive timestamps.
- Checking for gaps in the data.
- Counting records per day.


In [ ]:
# Analyze frequency of timestamps for Halifax
halifax_data = df[df['region'] == 'Halifax']

# Calculate time differences between consecutive timestamps
halifax_data['time_diff'] = halifax_data.index.to_series().diff().dt.total_seconds() / 3600

# Display the distribution of time differences
print("Time Difference Distribution (in hours):")
print(halifax_data['time_diff'].describe())

# Check for gaps in the data
gaps = halifax_data[halifax_data['time_diff'] > 1]
if not gaps.empty:
    print(f"Gaps found in the data:")
    print(gaps[['time_diff']])
else:
    print("No gaps found. Data appears to be hourly.")

# Count records per day
halifax_data['date'] = halifax_data.index.date
records_per_day = halifax_data.groupby('date').size()
print("Records per day:")
print(records_per_day.head())


In [ ]:
### Halifax Data Frequency Analysis Results
- **Time Frequency**: Appears to be hourly.
- **Gaps**: No gaps were found in the data; it seems consistent.
- **Records per Day**: The data contains consistent records for each day.


In [ ]:
## Preparing Data for Prophet Model
This section prepares the Halifax data for the Prophet model. The required format includes:
- `ds`: Datetime column.
- `y`: Target variable (e.g., `consumption_kwh`).
